# Week 2
## 1.Work flow Continue

In [4]:
#2026-5-14
# This block is to import the result and workflow last time.
import numpy as np
import pandas as pd
from scipy.fftpack import dct, idct
from pathlib import Path
import matplotlib.pyplot as plt
from kilosort.io import load_ops

SAVE_PATH = Path('F:\\ACADEMIC') / '.test_data' / 'ZFM-02370_mini.imec0.ap.short.bin'

n_chan = 385          # channel number
dtype = 'int16'

# Load the saved Sorting results from the local drive.
results_dir = SAVE_PATH.parent / 'kilosort4'
ops = load_ops(results_dir / 'ops.npy')
camps = pd.read_csv(results_dir / 'cluster_Amplitude.tsv', sep='\t')['Amplitude'].values
contam_pct = pd.read_csv(results_dir / 'cluster_ContamPct.tsv', sep='\t')['ContamPct'].values
chan_map =  np.load(results_dir / 'channel_map.npy')
templates =  np.load(results_dir / 'templates.npy')
chan_best = (templates**2).sum(axis=1).argmax(axis=-1)
chan_best = chan_map[chan_best]
amplitudes = np.load(results_dir / 'amplitudes.npy')
st = np.load(results_dir / 'spike_times.npy')
clu = np.load(results_dir / 'spike_clusters.npy')

fs = ops['fs']
firing_rates = np.unique(clu, return_counts=True)[1] * fs / st.max()
dshift = ops['dshift']

## 2. DCT for original

In [5]:
def compress_signal_dct(signal, keep_ratio=0.1): #signal: input singal, e.g. a part of data in a certain channel.
    
    n = len(signal)
    dct_coeffs = dct(signal, norm='ortho')
    
    cutoff = int(n * keep_ratio) # the number of remaining parameters
    compressed_coeffs = np.zeros(n)
    compressed_coeffs[:cutoff] = dct_coeffs[:cutoff]
    
    # inverse DCT to recover the signal
    reconstructed_signal = idct(compressed_coeffs, norm='ortho')
    
    return reconstructed_signal, compressed_coeffs